# PCM interactive

This notebook generates interactive plots to study Wangler's particle-core model (PCM).

In [ ]:
import argparse

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from ipywidgets import interact

from tools import pcm
from tools.plot import plot_rms_ellipse

plt.style.use("style.mplstyle")

## Simulation parameters

In [ ]:
# Simulation parameters
eta = 0.5  # tune depression ratio (k / k0)
mu = 0.5  # core mismatch factor (R / R0)
n = 50  # number of particles

# Package envelope parameters (R, R').
envelope = np.array([mu, 0.0])

# Distribute test particles along x axis.
particles = np.zeros((n, 2))
particles[:, 0] = np.linspace(0.0, 3.5, particles.shape[0])

## Continuous time evolution

In [ ]:
history = pcm.track(envelope, particles, eta=eta, t_max=134.0, t_steps=1000)

Plot beam size and particle position vs. time.

In [ ]:
@interact(index=(0, particles.shape[0] - 1))
def plot_x(index):
    fig, ax = plt.subplots(figsize=(7.0, 2.0))
    ax.fill_between(history["t"], -history["r"], +history["r"], color="black", alpha=0.2, ec="none")
    ax.plot(history["t"], history["particles"][:, index, 0], color="black")
    ax.set_ylim(-3.5, 3.5)
    ax.set_xlabel(r"$k_0 s$")
    ax.set_ylabel(r"$x / r_0$")
    plt.show()

Plot particle trajectory in phase space. Overlay envelope trajectory ($R$, $R'$).

In [ ]:
@interact(index=(0, particles.shape[0] - 1))
def plot_x_xp(index: int) -> None:
    fig, ax = plt.subplots(figsize=(3.5, 3.5))
    ax.plot(
        history["particles"][:, index, 0],
        history["particles"][:, index, 1],
        color="black",
        lw=0.5
    )
    for sign in [-1.0, +1.0]:
        ax.plot(
            history["r"] * sign,
            history["rp"],
            color="red",
            lw=0.5,
            alpha=0.1,

        )
    ax.set_xlabel(r"$x / r_0$")
    ax.set_ylabel(r"$x' / k_0 r_0$")
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    plt.show()

Plot particle trajectory in phase space up to a specified time.

In [ ]:
@interact(index=(0, particles.shape[0] - 1), steps=(1, history["particles"].shape[0] - 1))
def plot_x_xp(index: int, steps: int) -> None:
    fig, ax = plt.subplots(figsize=(3.5, 3.5))

    for sign in [-1.0, +1.0]:
        ax.plot(
            history["r"][:steps] * sign, 
            history["rp"][:steps], 
            color="red", 
            lw=0.5, 
            alpha=0.2
        )
        ax.scatter(history["r"][steps - 1] * sign, history["rp"][steps - 1], color="red")

    ax.plot(
        history["particles"][:steps, index, 0],
        history["particles"][:steps, index, 1],
        color="grey",
        lw=0.5
    )
    ax.scatter(
        history["particles"][steps - 1, index, 0],
        history["particles"][steps - 1, index, 1],
        color="black",
        zorder=999,
    )
    ax.set_xlabel(r"$x / r_0$")
    ax.set_ylabel(r"$x' / k_0 r_0$")
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.annotate(r"$k_0 s = {:0.2f}$".format(history["t"][steps - 1]), xy=(0.05, 0.95), xycoords="axes fraction")
    plt.show()

## Stroboscopic 

In [ ]:
history_strobe = pcm.track_strobe(envelope, particles, eta=eta, periods=100)

Plot single-particle trajectory.

In [ ]:
@interact(index=(0, particles.shape[0] - 1))
def plot_x_xp(index: int) -> None:
    fig, ax = plt.subplots(figsize=(3.5, 3.5))

    cov_matrix = pcm.get_cov_matrix(envelope[0], envelope[1], eta)
    plot_rms_ellipse(cov_matrix, ax=ax, level=2.0, fill=True, alpha=0.1, ec="none")

    ax.scatter(
        history_strobe["particles"][:, index, 0],
        history_strobe["particles"][:, index, 1],
        color="black",
        ec="none",
        s=1,
    )
    ax.set_xlabel(r"$x / r_0$")
    ax.set_ylabel(r"$x' / k_0 r_0$")
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    plt.show()

Plot all particle trajectories (one color).

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.scatter(
    history_strobe["particles"][..., 0],
    history_strobe["particles"][..., 1],
    ec="none",
    s=0.5,
    c="black",
)
ax.set_xlabel(r"$x / r_0$")
ax.set_ylabel(r"$x' / k_0 r_0$")
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
plt.show()

Plot all particle trajectories (multi-color).

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 3.5))
for index in range(0, history["particles"].shape[1], 1):
    ax.scatter(
        history_strobe["particles"][:, index, 0],
        history_strobe["particles"][:, index, 1],
        ec="none",
        s=0.5,
    )
ax.set_xlabel(r"$x / r_0$")
ax.set_ylabel(r"$x' / k_0 r_0$")
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
plt.show()